# Household Task Agent

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/rsasaki0109/PythonInteractiveRobotics/blob/main/notebooks/household_task_agent.ipynb)

A small household task loop: ambiguous command -> ask -> plan -> safety check -> grasp retry -> human correction -> replan -> place.

In [ ]:
from pathlib import Path
import os
import subprocess
import sys

ROOT = Path.cwd()
if not (ROOT / "pyproject.toml").exists():
    ROOT = Path("/content/PythonInteractiveRobotics")
    if not ROOT.exists():
        subprocess.run([
            "git",
            "clone",
            "--depth",
            "1",
            "https://github.com/rsasaki0109/PythonInteractiveRobotics.git",
            str(ROOT),
        ], check=True)

os.chdir(ROOT)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "."], check=True)
print(f"ready: {ROOT}")

In [ ]:
import importlib.util
import sys
from pathlib import Path

def load_example(relative_path: str):
    path = Path(relative_path)
    spec = importlib.util.spec_from_file_location(path.stem, path)
    module = importlib.util.module_from_spec(spec)
    sys.modules[spec.name] = module
    assert spec.loader is not None
    spec.loader.exec_module(module)
    return module

module = load_example("examples/embodied_ai/36_household_task_agent.py")
trace = module.run(command="put the block away", answer="red", seed=0, render=False)
summary = trace.summary()
print(summary)
print("failure kinds:", summary.failure_kinds)
print("final info:", summary.final_info)

In [ ]:
from IPython.display import Image, display

display(Image(filename="docs/assets/gifs/household_task_agent.gif"))

In [ ]:
# Compare ambiguous and explicit commands.
scenarios = [
    ("put the block away", "red"),
    ("put the blue block away", "red"),
]
for command, answer in scenarios:
    trace = module.run(command=command, answer=answer, seed=0, render=False)
    final = trace.infos[-1]
    print(
        f"command={command!r} stored={final['stored_color']} "
        f"questions={final['question_count']} replans={final['replan_count']} "
        f"failures={[f.kind for f in trace.failures()]}"
    )